**Modélisation du questionnaire:choix entre un pull neuf et un pull de seconde main**

NB : le produit choisi est un pull. Les prix ont été adaptés à ce produit :
- Neuf : 40, 60, 80 CHF (fast fashion, marque moyenne, bonne marque)
- Seconde main : 20, 40 CHF (particulier, magasin professionnel)

Nous sommes partis du principe qu'un prix plus élevé pour la seconde main ne ferait pas de sens sauf si c'est un pull de marque. Comme ici on veut tester les répondants sur un pull basique nous préférons rester sur des prix auxquels nous pensons que des jeunes serait prêt à payer un pull de seconde main.

Neuf : 3 attributs
- Prix : 40, 60, 80 CHF
- Unicité : 1 (Courant), 2 (Pièce unique)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

Seconde main : 4 attributs
- Prix : 20, 40 CHF
- Qualité : 1 (Bon état, traces d'usure visibles), 2 (Comme neuf)
- Unicité : 1 (Courant), 2 (Pièce unique)
- Garantie : 1 (Sans retour), 2 (Retours 30j)

**A) créations des profils**

In [1]:
import itertools
import pandas as pd
import numpy as np


In [2]:
# Définition des attributs PAR ALTERNATIVE (design labellisé : Neuf vs Seconde main)

# ---- Alternative "Neuf" : 3 attributs ----
prix_neuf_niveaux = [40, 60, 80]
unicite_neuf_niveaux = [1, 2]   # 1: Standard de collection courante, 2: Pièce unique/vintage introuvable en magasin
garantie_neuf_niveaux = [1, 2]     # 1: Sans retour, 2: Retours 30j

profils_neuf = list(itertools.product(
    prix_neuf_niveaux, unicite_neuf_niveaux, garantie_neuf_niveaux
))
df_profils_neuf = pd.DataFrame(
    profils_neuf, columns=['Prix', 'Unicité', 'Garantie']
)

# ---- Alternative "Seconde main" : 4 attributs (Qualité est spécifique à la SM) ----
prix_sm_niveaux = [20, 40]
qualite_sm_niveaux = [1, 2]        # 1: Bon état avec des traces d'usure visibles, 2: Comme neuf
unicite_sm_niveaux = [1, 2]    # 1: Standard de collection courante, 2: Pièce unique/vintage introuvable en magasin
garantie_sm_niveaux = [1, 2]       # 1: Sans retour, 2: Retours 30j

profils_sm = list(itertools.product(
    prix_sm_niveaux, qualite_sm_niveaux, unicite_sm_niveaux, garantie_sm_niveaux
))
df_profils_sm = pd.DataFrame(
    profils_sm, columns=['Prix', 'Qualité', 'Unicité', 'Garantie']
)

print(f"Profils Neuf (plein factoriel) : {len(df_profils_neuf)}")
print(df_profils_neuf.to_string(index=False))
print()
print(f"Profils Seconde main (plein factoriel) : {len(df_profils_sm)}")
print(df_profils_sm.to_string(index=False))

Profils Neuf (plein factoriel) : 12
 Prix  Unicité  Garantie
   40        1         1
   40        1         2
   40        2         1
   40        2         2
   60        1         1
   60        1         2
   60        2         1
   60        2         2
   80        1         1
   80        1         2
   80        2         1
   80        2         2

Profils Seconde main (plein factoriel) : 16
 Prix  Qualité  Unicité  Garantie
   20        1        1         1
   20        1        1         2
   20        1        2         1
   20        1        2         2
   20        2        1         1
   20        2        1         2
   20        2        2         1
   20        2        2         2
   40        1        1         1
   40        1        1         2
   40        1        2         1
   40        1        2         2
   40        2        1         1
   40        2        1         2
   40        2        2         1
   40        2        2         2


**B) Question a insérer dans le questionnaires**

créations de toutes les paires candidates:

In [3]:
from itertools import product

In [4]:
# Générer toutes les paires : seconde main vs neuf
paires = []

for (idx_sm, profil_sm), (idx_neuf, profil_neuf) in product(
    df_profils_sm.iterrows(),
    df_profils_neuf.iterrows()
):
    paires.append({
        "Prix_SM": profil_sm["Prix"],
        "Qualite_SM": profil_sm["Qualité"],
        "Unicite_SM": profil_sm["Unicité"],
        "Retours_SM": profil_sm["Garantie"],
        "Prix_neuf": profil_neuf["Prix"],
        "Unicite_neuf": profil_neuf["Unicité"],
        "Retours_neuf": profil_neuf["Garantie"]
    })

df_paires_candidates = pd.DataFrame(paires)

print(f"Nombre de paires candidates : {len(df_paires_candidates)}")
print(df_paires_candidates.to_string(index=False))


Nombre de paires candidates : 192
 Prix_SM  Qualite_SM  Unicite_SM  Retours_SM  Prix_neuf  Unicite_neuf  Retours_neuf
      20           1           1           1         40             1             1
      20           1           1           1         40             1             2
      20           1           1           1         40             2             1
      20           1           1           1         40             2             2
      20           1           1           1         60             1             1
      20           1           1           1         60             1             2
      20           1           1           1         60             2             1
      20           1           1           1         60             2             2
      20           1           1           1         80             1             1
      20           1           1           1         80             1             2
      20           1           1          

In [5]:
# Supprimer les questions évidentes :
# une option est au moins aussi bonne sur le prix, l'unicité ET les retours
def est_evident(row):
    sm_mieux = (row["Prix_SM"] <= row["Prix_neuf"]
                and row["Unicite_SM"] >= row["Unicite_neuf"]
                and row["Retours_SM"] >= row["Retours_neuf"]
                )
    neuf_mieux = (row["Prix_neuf"] <= row["Prix_SM"]
                  and row["Unicite_neuf"] >= row["Unicite_SM"]
                  and row["Retours_neuf"] >= row["Retours_SM"])
    return sm_mieux or neuf_mieux

df_paires_candidates = (
    df_paires_candidates[~df_paires_candidates.apply(est_evident, axis=1)]
    .reset_index(drop=True)
)
print("Paires restantes :", len(df_paires_candidates))

Paires restantes : 74


**C) Choix des questions**
but min d-error

In [6]:
pip install choicedesign

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import choicedesign as cd

Design D-efficient avec priors supposés (pull)
- Prix linéaire : un seul coefficient par franc
- Retours séparés : un effet pour la seconde main, un pour le neuf
- Priors supposés (signe attendu), à remplacer par les estimations du questionnaire test

In [8]:
from choicedesign.design import EffDesign
from choicedesign.expressions import Attribute, Parameter, ASC

# --------------------------------------------------
# 1. Attributs
# --------------------------------------------------
prix_sm = Attribute("Prix_SM", prix_sm_niveaux)
qualite_sm = Attribute("Qualite_SM", qualite_sm_niveaux)
unicite_sm = Attribute("Unicite_SM", unicite_sm_niveaux)
retours_sm = Attribute("Retours_SM", garantie_sm_niveaux)

prix_neuf = Attribute("Prix_neuf", prix_neuf_niveaux)
unicite_neuf = Attribute("Unicite_neuf", unicite_neuf_niveaux)
retours_neuf = Attribute("Retours_neuf", garantie_neuf_niveaux)

# --------------------------------------------------
# 2. Paramètres
# Prix LINÉAIRE : un seul coefficient "par franc"
# Priors supposés (signe attendu), pas tirés de la littérature
# --------------------------------------------------
beta_prix = Parameter("beta_prix", -0.0001)
beta_unicite = Parameter("beta_unicite", 0.0000)
beta_qualite = Parameter("beta_qualite", 0.0001)      # signe négatif, quasi nul
beta_retours_sm = Parameter("beta_retours_sm", 0.0001) # signe positif, quasi nul
beta_retours_neuf = Parameter("beta_retours_neuf", 0.0001)
asc_sm = ASC("asc_sm", 0.0)  # déjà à 0


# --------------------------------------------------
# 3. Fonctions d'utilité
# Le prix entre directement comme un nombre
# --------------------------------------------------
V_sm = (
    asc_sm
    + beta_prix * prix_sm
    + beta_qualite * (qualite_sm == 2)
    + beta_unicite * (unicite_sm == 2)
    + beta_retours_sm * (retours_sm == 2)
)

V_neuf = (
    beta_prix * prix_neuf
    + beta_unicite * (unicite_neuf == 2)
    + beta_retours_neuf * (retours_neuf == 2)
)

V = {
    1: V_sm,
    2: V_neuf
}

candidats:

In [9]:
# --------------------------------------------------
# 4. Préparer les 144 paires candidates
# --------------------------------------------------

colonnes_modele = [
    "Prix_SM",
    "Qualite_SM",
    "Unicite_SM",
    "Retours_SM",
    "Prix_neuf",
    "Unicite_neuf",
    "Retours_neuf"
]

candidates = df_paires_candidates.reset_index(drop=True).copy()


# --------------------------------------------------
# 5. Créer l'évaluateur D-efficient
# --------------------------------------------------

design = EffDesign(
    X=[
        prix_sm,
        qualite_sm,
        unicite_sm,
        retours_sm,
        prix_neuf,
        unicite_neuf,
        retours_neuf
    ],
    ncs=12
)


def evaluer_selection(indices):
    plan = candidates.iloc[indices][colonnes_modele].copy()

    # Calcul de la pénalité si la condition n'est pas respectée
    nb_standard = (plan["Qualite_SM"] == 1).sum()
    penalite = 0.0
    if nb_standard < 5:
        # Pénalité douce : plus il manque de profils standard, plus la D-error sera élevée
        penalite = (5 - nb_standard) * 10.0

    plan.insert(0, "CS", range(1, len(plan) + 1))

    d_error, equilibre = design.evaluate(
        plan,
        V=V,
        criterion="d"
    )

    # On ajoute la pénalité à la D-error trouvée
    return float(d_error) + penalite, float(equilibre)

# --------------------------------------------------
# 6. Recherche locale avec plusieurs départs aléatoires
#    (version limitée en temps : échantillonnage des candidats testés à chaque swap)
# --------------------------------------------------

rng = np.random.default_rng(42)

meilleur_plan = None
meilleur_d_error = np.inf
meilleur_equilibre = None

nombre_depart = 5
iterations_max = 25
taille_echantillon = 70

for depart in range(nombre_depart):

    # tirer un départ qui respecte déjà la règle des 4 "standard"
    while True:
        selection = list(
            rng.choice(len(candidates), size=12, replace=False)
        )
        if (candidates.iloc[selection]["Qualite_SM"] == 1).sum() >= 5:
            break

    d_error, equilibre = evaluer_selection(selection)

    for iteration in range(iterations_max):

        indices_selectionnes = set(selection)
        candidats_restants = [
            i for i in range(len(candidates)) if i not in indices_selectionnes
        ]
        candidats_testes = rng.choice(
            candidats_restants,
            size=min(taille_echantillon, len(candidats_restants)),
            replace=False
        )

        meilleur_echange = None
        meilleur_d_local = d_error
        meilleur_eq_local = equilibre

        for position in range(12):

            for candidate_idx in candidats_testes:

                essai = selection.copy()
                essai[position] = candidate_idx

                d_essai, eq_essai = evaluer_selection(essai)

                if (
                    np.isfinite(d_essai)
                    and d_essai < meilleur_d_local - 1e-10
                ):
                    meilleur_d_local = d_essai
                    meilleur_eq_local = eq_essai
                    meilleur_echange = essai

        if meilleur_echange is None:
            break

        selection = meilleur_echange
        d_error = meilleur_d_local
        equilibre = meilleur_eq_local

    print(f"Départ {depart + 1}/{nombre_depart} terminé — D-error : {d_error:.4f}")

    if d_error < meilleur_d_error:
        meilleur_d_error = d_error
        meilleur_equilibre = equilibre
        meilleur_plan = selection.copy()

# --------------------------------------------------
# 7. Afficher les 12 questions sélectionnées
# --------------------------------------------------

df_questions = candidates.iloc[meilleur_plan].copy()
df_questions = df_questions.reset_index(drop=True)

df_questions.insert(
    0, "Question", range(1, len(df_questions) + 1)
)

print("Nombre de questions :", len(df_questions))
print("D-error final :", meilleur_d_error)
print("Équilibre des utilités (%) :", meilleur_equilibre)

display(df_questions)

Départ 1/5 terminé — D-error : 0.1497
Départ 2/5 terminé — D-error : 0.1497
Départ 3/5 terminé — D-error : 0.1497
Départ 4/5 terminé — D-error : 0.1497
Départ 5/5 terminé — D-error : 0.1497
Nombre de questions : 12
D-error final : 0.14971754922186728
Équilibre des utilités (%) : 99.99951141945495


,Question,Prix_SM,Qualite_SM,Unicite_SM,Retours_SM,Prix_neuf,Unicite_neuf,Retours_neuf
0,1,20,2,1,1,80,2,1
1,2,40,2,1,2,40,2,1
2,3,20,1,2,1,80,1,2
3,4,40,2,2,1,40,1,2
4,5,20,2,1,2,80,2,1
5,6,40,2,1,1,60,2,2
6,7,20,1,1,1,80,2,2
7,8,20,2,1,1,40,2,2
8,9,20,1,1,2,80,2,1
9,10,40,1,1,2,60,2,2


In [10]:
# Dictionnaires de correspondance code -> libellé
labels_qualite = {1: "Bon état, avec traces d'ususre visible", 2: "Comme neuf"}
labels_unicite = {1: "Pièce mainstream", 2: "Trouvaille unique"}
labels_garantie = {1: "Sans retour", 2: "Retours 30j"}

df_questions_lisible = df_questions.copy()

df_questions_lisible["Qualite_SM"] = df_questions_lisible["Qualite_SM"].replace(labels_qualite)
df_questions_lisible["Unicite_SM"] = df_questions_lisible["Unicite_SM"].replace(labels_unicite)
df_questions_lisible["Retours_SM"] = df_questions_lisible["Retours_SM"].replace(labels_garantie)
df_questions_lisible["Unicite_neuf"] = df_questions_lisible["Unicite_neuf"].replace(labels_unicite)
df_questions_lisible["Retours_neuf"] = df_questions_lisible["Retours_neuf"].replace(labels_garantie)

display(df_questions_lisible)

,Question,Prix_SM,Qualite_SM,Unicite_SM,Retours_SM,Prix_neuf,Unicite_neuf,Retours_neuf
0,1,20,Comme neuf,Pièce mainstream,Sans retour,80,Trouvaille unique,Sans retour
1,2,40,Comme neuf,Pièce mainstream,Retours 30j,40,Trouvaille unique,Sans retour
2,3,20,"Bon état, avec traces d'ususre visible",Trouvaille unique,Sans retour,80,Pièce mainstream,Retours 30j
3,4,40,Comme neuf,Trouvaille unique,Sans retour,40,Pièce mainstream,Retours 30j
4,5,20,Comme neuf,Pièce mainstream,Retours 30j,80,Trouvaille unique,Sans retour
5,6,40,Comme neuf,Pièce mainstream,Sans retour,60,Trouvaille unique,Retours 30j
6,7,20,"Bon état, avec traces d'ususre visible",Pièce mainstream,Sans retour,80,Trouvaille unique,Retours 30j
7,8,20,Comme neuf,Pièce mainstream,Sans retour,40,Trouvaille unique,Retours 30j
8,9,20,"Bon état, avec traces d'ususre visible",Pièce mainstream,Retours 30j,80,Trouvaille unique,Sans retour
9,10,40,"Bon état, avec traces d'ususre visible",Pièce mainstream,Retours 30j,60,Trouvaille unique,Retours 30j


**Choix de modélisation retenu**

- Prix linéaire : un seul coefficient par franc (beta_prix). Permet de calculer la disposition à payer (ex. : combien de CHF vaut un vêtement avec un état comme neuf).
- Retours séparés : beta_retours_sm et beta_retours_neuf. Hypothèse : la possibilité de retour rassure davantage pour un pull de seconde main (risque sur l'état, la propreté, la taille) que pour un pull neuf. Séparer les deux permet de tester cette hypothèse statistiquement.
- Unicité des pièces : un seul beta_unicite pour les deux alternatives (pas de raison claire de penser que l'effet diffère).
- Qualité: seulement pour la seconde main (un pull neuf est forcément comme neuf).
- Questions évidentes retirées : 74 paires candidates restantes, toutes avec un vrai arbitrage.


In [11]:
import pandas as pd
from IPython.display import display, HTML

# Parcourir les 12 questions
for idx, row in df_questions_lisible.iterrows():
    num_q = int(row['Question'])

    # Données pour SM et Neuf
    col_sm = [
        f"{row['Prix_SM']} CHF",
        row.get('Qualite_SM'),
        row.get('Unicite_SM'),
        row.get('Retours_SM')
    ]

    col_neuf = [
        f"{row['Prix_neuf']} CHF",
        row.get('Qualite_neuf', 'Neuf'),
        row.get('Unicite_neuf'),
        row.get('Retours_neuf')
    ]

    # Alterner le sens gauche/droite pour l'effet A / B
    inverser = (num_q % 2 == 0)

    if inverser:
        data_paire = {"Neuf": col_neuf, "Seconde Main": col_sm}
    else:
        data_paire = {"Seconde Main": col_sm, "Neuf": col_neuf}

    df_q = pd.DataFrame(data_paire, index=['Prix', 'Qualité', 'Unicité', 'Retours'])

    # Application du style : fond blanc pur, lignes horizontales pour chaque attribut, et trait vertical très net
    styled_df = df_q.style \
        .set_caption(f"<b>Question {num_q} / 12</b>") \
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '10px'), ('color', '#2c3e50')]},
            {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '30px'), ('width', '100%'), ('max-width', '600px'), ('font-family', 'sans-serif'), ('background-color', '#ffffff'), ('box-shadow', '0 4px 6px rgba(0,0,0,0.05)')]},
            {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('font-weight', '700'), ('font-size', '14px'), ('padding', '10px 14px'), ('text-align', 'center'), ('border-bottom', '2px solid #cbd5e1')]},
            {'selector': 'td', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('padding', '10px 14px'), ('border-bottom', '1px solid #e2e8f0'), ('text-align', 'center'), ('font-size', '13px')]},

            # Colonne des attributs (gauche) : fond blanc et lignes horizontales conservées, avec un trait vertical très net à droite
            {'selector': 'th.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('width', '100px'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '2px solid #cbd5e1')]},
            {'selector': 'td.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '1px solid #e2e8f0')]},

            # Fermer le bloc tout en haut à gauche
            {'selector': 'thead tr th:first-child', 'props': [('border-right', '3px solid #1e293b'), ('background-color', '#ffffff')]},

            # Ligne de séparation entre l'Alternative A et l'Alternative B (milieu)
            {'selector': 'td:nth-child(2), th:nth-child(2)', 'props': [('border-right', '2px solid #cbd5e1')]}
        ])

    display(styled_df)

,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Sans retour,Sans retour


,Neuf,Seconde Main
Prix,40 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Neuf,Seconde Main
Prix,40 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Neuf,Seconde Main
Prix,60 CHF,40 CHF
Qualité,Neuf,Comme neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Sans retour


... (2 outputs left out)

,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Retours 30j,Sans retour


,Neuf,Seconde Main
Prix,60 CHF,40 CHF
Qualité,Neuf,"Bon état, avec traces d'ususre visible"
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Retours 30j


,Seconde Main,Neuf
Prix,20 CHF,80 CHF
Qualité,Comme neuf,Neuf
Unicité,Trouvaille unique,Pièce mainstream
Retours,Sans retour,Retours 30j


,Neuf,Seconde Main
Prix,40 CHF,20 CHF
Qualité,Neuf,"Bon état, avec traces d'ususre visible"
Unicité,Trouvaille unique,Pièce mainstream
Retours,Retours 30j,Retours 30j


question piège

In [12]:
# 1. Création de la question piège
question_piege = pd.DataFrame([{
    'Question': 99,  # Numéro distinct pour l'identifier facilement
    'Prix_SM': 60,
    'Qualite_SM': "Bon état, avec traces d'ususre visible",
    'Unicite_SM': "Pièce mainstream",
    'Retours_SM': "Sans retour",
    'Prix_neuf': 40,
    'Qualite_neuf': "Neuf",
    'Unicite_neuf': "Trouvaille unique",
    'Retours_neuf': "Retours 30j"
}])

# 2. Préparation des données pour l'affichage
row = question_piege.iloc[0]
num_q = int(row['Question'])

col_sm = [
    f"{row['Prix_SM']} CHF",
    row.get('Qualite_SM'),
    row.get('Unicite_SM'),
    row.get('Retours_SM')
]

col_neuf = [
    f"{row['Prix_neuf']} CHF",
    row.get('Qualite_neuf', 'Neuf'),
    row.get('Unicite_neuf'),
    row.get('Retours_neuf')
]

# Alternative A : Seconde Main / Alternative B : Neuf
data_paire = {
    "Seconde Main": col_sm,
    "Neuf": col_neuf
}

df_q = pd.DataFrame(data_paire, index=['Prix', 'Qualité', 'Unicité', 'Retours'])

# 3. Affichage avec votre design validé
styled_df = df_q.style \
    .set_caption("<b>Question Piège</b> <span style='color: #e74c3c; font-weight: normal;'>(Choix évident : Neuf)</span>") \
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('text-align', 'left'), ('margin-bottom', '10px'), ('color', '#2c3e50')]},
        {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('margin-bottom', '30px'), ('width', '100%'), ('max-width', '600px'), ('font-family', 'sans-serif'), ('background-color', '#ffffff'), ('box-shadow', '0 4px 6px rgba(0,0,0,0.05)')]},
        {'selector': 'th', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('font-weight', '700'), ('font-size', '14px'), ('padding', '10px 14px'), ('text-align', 'center'), ('border-bottom', '2px solid #cbd5e1')]},
        {'selector': 'td', 'props': [('background-color', '#ffffff'), ('color', '#1e293b'), ('padding', '10px 14px'), ('border-bottom', '1px solid #e2e8f0'), ('text-align', 'center'), ('font-size', '13px')]},

        # Colonne des attributs (gauche) : fond blanc, lignes horizontales, trait vertical très net à droite
        {'selector': 'th.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('width', '100px'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '2px solid #cbd5e1')]},
        {'selector': 'td.row_heading', 'props': [('background-color', '#ffffff'), ('color', '#0f172a'), ('font-weight', '600'), ('text-align', 'left'), ('border-right', '3px solid #1e293b'), ('border-bottom', '1px solid #e2e8f0')]},

        # Fermer le bloc tout en haut à gauche
        {'selector': 'thead tr th:first-child', 'props': [('border-right', '3px solid #1e293b'), ('background-color', '#ffffff')]},

        # Ligne de séparation entre l'Alternative A et l'Alternative B (milieu)
        {'selector': 'td:nth-child(2), th:nth-child(2)', 'props': [('border-right', '2px solid #cbd5e1')]}
    ])

display(styled_df)

,Seconde Main,Neuf
Prix,60 CHF,40 CHF
Qualité,"Bon état, avec traces d'ususre visible",Neuf
Unicité,Pièce mainstream,Trouvaille unique
Retours,Sans retour,Retours 30j


In [13]:
# Sauvegarde du design optimal pour l'utiliser plus tard
df_questions.to_csv("mon_design_pretest.csv", index=False)